# Kaggle LLM Server — llama.cpp + CUDA

Builds llama.cpp from source with CUDA (fixes `CUDA::cuda_driver` target issue on Kaggle).  
Automatically falls back to the latest pre-built release binary if the source build fails.

### Before running
1. **Runtime → Change runtime type** → select **GPU (T4 or P100)**
2. Set your `HF_TOKEN` in Kaggle Secrets if the model repo is gated
3. Adjust `MODEL_URL` / `MODEL_NAME` below as needed

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import os

MODEL_URL  = "https://huggingface.co/HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive/resolve/main/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"
MODEL_NAME = "Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"
MODEL_DIR  = "/kaggle/working/models"
LLAMA_DIR  = "/kaggle/working/llama.cpp"
HF_TOKEN   = os.environ.get("HF_TOKEN", "")

# Kaggle T4 → sm_75; P100 → sm_60; A100 → sm_80
CUDA_ARCH  = "75"

# Pre-built release to use when source build fails
PREBUILT_RELEASE = "b4780"
PREBUILT_URL = f"https://github.com/ggml-org/llama.cpp/releases/download/{PREBUILT_RELEASE}/llama-{PREBUILT_RELEASE}-bin-ubuntu-x64.zip"

print(f"Model : {MODEL_NAME}")
print(f"HF token set: {bool(HF_TOKEN)}")

In [ ]:
# ── Cell 1: Build llama.cpp from source with CUDA ────────────────────────────
#
# Root cause of the `CUDA::cuda_driver` CMake error on Kaggle:
#   CMake's CUDAToolkit module looks for libcuda.so.1 to create the
#   CUDA::cuda_driver IMPORTED target, but Kaggle only ships the stub
#   at /usr/local/cuda/lib64/stubs/libcuda.so (no .1 suffix, not in
#   the default search path).  A single symlink fixes it.

import subprocess, shutil

BUILD_SUCCESS = False
SERVER_BIN    = None

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd))
    return subprocess.run(cmd, check=True, **kw)

# ── Fix: symlink CUDA stub so CMake finds CUDA::cuda_driver ──────────────────
cuda_stub  = "/usr/local/cuda/lib64/stubs/libcuda.so"
cuda_link  = "/usr/local/cuda/lib64/libcuda.so.1"
if os.path.exists(cuda_stub) and not os.path.exists(cuda_link):
    try:
        os.symlink(cuda_stub, cuda_link)
        print(f"Linked {cuda_stub} → {cuda_link}")
    except OSError as e:
        print(f"symlink failed (non-fatal): {e}")

os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/cuda/lib64/stubs:"
    + os.environ.get("LD_LIBRARY_PATH", "")
    + ":/usr/local/cuda/lib64"
)

# ── Install build tools ───────────────────────────────────────────────────────
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ninja-build", "cmake", "git", "wget", "unzip"])

# ── Clone llama.cpp ───────────────────────────────────────────────────────────
if os.path.exists(LLAMA_DIR):
    shutil.rmtree(LLAMA_DIR)
run(["git", "clone", "--depth=1", "https://github.com/ggml-org/llama.cpp", LLAMA_DIR])

# ── CMake configure + build ───────────────────────────────────────────────────
build_dir = f"{LLAMA_DIR}/build"
try:
    run([
        "cmake", "-B", build_dir, "-GNinja", "-S", LLAMA_DIR,
        "-DGGML_CUDA=ON",
        "-DGGML_CUDA_NO_PEER_COPY=ON",   # safe on single-GPU Kaggle
        "-DGGML_NATIVE=OFF",
        f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}",
        "-DCMAKE_BUILD_TYPE=Release",
    ])
    run(["cmake", "--build", build_dir, "--config", "Release", "-j4"])

    candidate = os.path.join(build_dir, "bin", "llama-server")
    if not os.path.exists(candidate):
        # Some builds place it directly in build/
        result = subprocess.run(
            ["find", build_dir, "-name", "llama-server", "-type", "f"],
            capture_output=True, text=True
        )
        candidate = result.stdout.strip().splitlines()[0] if result.stdout.strip() else ""

    if candidate and os.path.exists(candidate):
        SERVER_BIN    = candidate
        BUILD_SUCCESS = True
        print(f"\nSource build succeeded → {SERVER_BIN}")
    else:
        print("Build finished but llama-server not found — will fall back")

except subprocess.CalledProcessError as exc:
    print(f"\nSource build failed (exit {exc.returncode}) — falling back to pre-built binary")


In [ ]:
# ── Cell 2: Fallback — download pre-built binary ──────────────────────────────
# Runs only when the source build did not produce a usable llama-server.

if not BUILD_SUCCESS:
    zip_path    = "/tmp/llama_prebuilt.zip"
    extract_dir = f"{LLAMA_DIR}/prebuilt"
    os.makedirs(extract_dir, exist_ok=True)

    print(f"Downloading pre-built release {PREBUILT_RELEASE} …")
    run(["wget", "-q", "--show-progress", "-O", zip_path, PREBUILT_URL])
    run(["unzip", "-q", "-o", zip_path, "-d", extract_dir])

    # The zip unpacks to extract_dir/build/bin/llama-server or similar
    result = subprocess.run(
        ["find", extract_dir, "-name", "llama-server", "-type", "f"],
        capture_output=True, text=True
    )
    candidates = result.stdout.strip().splitlines()
    if not candidates:
        raise RuntimeError(
            f"llama-server not found in {extract_dir} after extracting {PREBUILT_URL}\n"
            "Check that the release URL is correct and the zip contains the binary."
        )

    SERVER_BIN = candidates[0]
    os.chmod(SERVER_BIN, 0o755)
    print(f"Pre-built binary ready → {SERVER_BIN}")

print(f"\nllama-server: {SERVER_BIN}")

In [ ]:
# ── Cell 3: Download model ────────────────────────────────────────────────────
import os

os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, MODEL_NAME)

min_size = 1_000_000_000  # 1 GB — anything smaller is a partial download
if os.path.exists(model_path) and os.path.getsize(model_path) >= min_size:
    print(f"Model already present ({os.path.getsize(model_path)/1e9:.1f} GB) — skipping download")
else:
    print(f"Downloading {MODEL_NAME} …")
    cmd = ["wget", "-q", "--show-progress", "-O", model_path]
    if HF_TOKEN:
        cmd += ["--header", f"Authorization: Bearer {HF_TOKEN}"]
    cmd.append(MODEL_URL)
    run(cmd)
    print(f"Downloaded → {model_path} ({os.path.getsize(model_path)/1e9:.1f} GB)")

In [ ]:
# ── Cell 4: Launch llama-server ───────────────────────────────────────────────
import subprocess, os

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1"

server_cmd = [
    SERVER_BIN,
    "-m",   model_path,
    "-ngl", "999",          # offload all layers to GPU
    "-sm",  "layer",        # split mode: distribute layers across GPUs
    "-ts",  "1,1",          # tensor split ratio (equal for dual-GPU)
    "-c",   "131072",       # context length
    "-n",   "4096",         # max new tokens per request
    "--temp",              "1.0",
    "--top-p",             "0.95",
    "--top-k",             "20",
    "--presence-penalty",  "1.5",
    "--jinja",
    "--host", "0.0.0.0",
    "--port", "8080",
]

print("Starting llama-server …")
print(" ".join(server_cmd))
process = subprocess.Popen(server_cmd, env=env)
try:
    process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    print("Server stopped.")

In [ ]:
# ── Cell 5: Quick smoke-test (run after server is up) ─────────────────────────
import requests, time

SERVER = "http://localhost:8080"

# Wait for the server to be ready
for _ in range(30):
    try:
        r = requests.get(f"{SERVER}/health", timeout=2)
        if r.status_code == 200:
            print("Server is ready")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    print("Server did not become ready in time — check Cell 4 output")

def ask(question, system="You are a helpful assistant.", temp=1.0):
    resp = requests.post(
        f"{SERVER}/v1/chat/completions",
        json={
            "messages": [
                {"role": "system",  "content": system},
                {"role": "user",    "content": question},
            ],
            "temperature":      temp,
            "top_p":            0.95,
            "top_k":            20,
            "presence_penalty": 1.5,
            "max_tokens":       4096,
            "stream":           False,
        },
        timeout=300,
    )
    return resp.json()["choices"][0]["message"]["content"]

print(ask("Hello, what can you do?"))